In [ ]:
from tensorflow.keras import backend as K
import gc

# Clear any previous model from memory
K.clear_session()

# Optionally, force garbage collection to free memory
gc.collect()

# Proceed with defining and training the new model


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error, mean_absolute_error
from pmdarima import auto_arima

# Load the data
data = pd.read_csv("../../../../data/vsf_prediction_model/demo/lar/lar_index.csv")
data['Date'] = pd.to_datetime(data['Date'], dayfirst=True)
data.set_index('Date', inplace=True)

# Extract LAR values
lar_values = data['LAR']

# Plot the data
plt.figure(figsize=(10, 6))
plt.plot(data.index, lar_values, label="LAR Index")
plt.title("LAR Historical Data")
plt.xlabel("Date")
plt.ylabel("LAR")
plt.legend()
plt.show()

# Split the data into training and testing sets
train_size = int(len(lar_values) * 0.8)
train, test = lar_values[:train_size], lar_values[train_size:]

# Find the best SARIMA parameters using auto_arima
auto_model = auto_arima(
    train, 
    seasonal=True, 
    m=12,  # Assuming monthly seasonality
    trace=True, 
    error_action="ignore", 
    suppress_warnings=True, 
    stepwise=True
)
print("Best SARIMA Order:", auto_model.order)
print("Best Seasonal Order:", auto_model.seasonal_order)

# Define and fit the SARIMA model with the best parameters
model = SARIMAX(train, order=auto_model.order, seasonal_order=auto_model.seasonal_order)
sarima_model = model.fit(disp=False)

# Forecast on the test set
test_forecast = sarima_model.forecast(steps=len(test))

# Evaluate the model
mse = mean_squared_error(test, test_forecast)
mae = mean_absolute_error(test, test_forecast)
print(f"Mean Squared Error (MSE): {mse}")
print(f"Mean Absolute Error (MAE): {mae}")

# Plot the forecast vs actual
plt.figure(figsize=(10, 6))
plt.plot(test.index, test, label="Actual")
plt.plot(test.index, test_forecast, label="SARIMA Forecast", color='red')
plt.legend()
plt.title("SARIMA Forecast vs Actual")
plt.show()

# Predict future values
future_steps = 365  # Number of days to predict
future_forecast = sarima_model.get_forecast(steps=future_steps)
future_index = pd.date_range(data.index[-1] + pd.Timedelta(days=1), periods=future_steps)
future_conf_int = future_forecast.conf_int()  # Confidence intervals

# Create a DataFrame for future predictions
future_df = pd.DataFrame({
    'Date': future_index,
    'Predicted_LAR': future_forecast.predicted_mean,
    'Lower_CI': future_conf_int.iloc[:, 0],
    'Upper_CI': future_conf_int.iloc[:, 1]
})

# Save the future predictions
future_df.to_csv("SARIMA_Predicted_LAR.csv", index=False)

# Plot future predictions
plt.figure(figsize=(10, 6))
plt.plot(data.index, lar_values, label="Historical LAR")
plt.plot(future_df['Date'], future_df['Predicted_LAR'], label="Future Forecast", color='red')
plt.fill_between(
    future_df['Date'], 
    future_df['Lower_CI'], 
    future_df['Upper_CI'], 
    color='pink', 
    alpha=0.3, 
    label="Confidence Interval"
)
plt.legend()
plt.title("Future LAR Forecast with SARIMA")
plt.show()
